# **Day 99: The Main GIS and Remote Sensing Libraries in Python**

In Days 90 to 94, we wrote GIS operations **ourselves** (projections, area, point-in-polygon, raster maths, classification) to understand how they work. In real projects, we use **libraries** that do all of this faster and more correctly.

This notebook is a **guide** to the most important libraries. The code cells are **examples to run after installing the libraries**, so they are not executed here.

**Installing (use conda for GIS, see Day 66):**
```bash
conda install -c conda-forge geopandas rasterio shapely pyproj folium scikit-learn
```
or
```bash
pip install geopandas rasterio shapely pyproj folium scikit-learn
```

## **The GIS Python Family**
| Library | Used for | Similar to what we did on |
|---|---|---|
| **Shapely** | Geometry operations (area, buffer, intersection) | Day 91 |
| **PyProj** | Coordinate system conversion | Day 90 |
| **GeoPandas** | Vector data tables (like Pandas + geometry) | Days 83, 91 |
| **Rasterio** | Reading and writing raster files (GeoTIFF) | Day 92 |
| **NumPy** | Raster maths (NDVI etc.) | Days 81, 82 |
| **scikit-learn** | Machine learning classification | Day 93 |
| **Folium** | Interactive web maps | Day 96 |
| **Google Earth Engine (geemap)** | Satellite data analysis in the cloud | Days 93, 94 |

Almost all of these are built on two old and powerful C libraries: **GDAL** (raster and vector file formats) and **GEOS** (geometry).

## **1. Shapely: Geometry Operations**
Everything from Day 91 becomes one line each.

In [ ]:
from shapely.geometry import Point, LineString, Polygon

field = Polygon([(0, 0), (100, 0), (100, 50), (0, 50)])       # In metres
well = Point(120, 40)
road = LineString([(0, 0), (300, 0), (300, 400)])

print(field.area)                  # 5000.0 (Shoelace formula inside)
print(road.length)                 # 700.0
print(field.centroid)              # POINT (50 25)
print(field.contains(Point(10, 10)))   # True (point in polygon)
print(well.distance(field))        # 20.0 (point to polygon distance)

buffer_zone = road.buffer(50)      # A 50 m buffer around the road
print(buffer_zone.intersects(field))
print(field.intersection(buffer_zone).area)

## **2. PyProj: Converting Coordinates**
Day 90's long UTM formula becomes two lines, and it works for **any** of thousands of coordinate systems.

In [ ]:
from pyproj import Transformer

to_utm = Transformer.from_crs("EPSG:4326", "EPSG:32645", always_xy=True)
x, y = to_utm.transform(88.3639, 22.5726)         # Note: always_xy=True means (lon, lat) order
print(x, y)                                        # About 640220, 2496849 (same as our Day 90 formula)

to_web = Transformer.from_crs("EPSG:4326", "EPSG:3857", always_xy=True)
print(to_web.transform(88.3639, 22.5726))

## **3. GeoPandas: Tables with Geometry**
A **GeoDataFrame** is a Pandas DataFrame with a special **geometry** column. It can read Shapefiles, GeoJSON and GeoPackages directly.

In [ ]:
import geopandas as gpd

# Read a GeoJSON file (for example the one we made on Day 40 or Day 98)
points = gpd.read_file("wb_weekly_weather.geojson")
print(points.crs)                     # EPSG:4326
print(points.head())

# Reproject to UTM and make 10 km buffers around every HQ
points_utm = points.to_crs(epsg=32645)
points_utm["buffer_10km"] = points_utm.buffer(10_000)

# Read district boundaries (a Shapefile or GeoJSON downloaded from a government portal)
districts = gpd.read_file("wb_districts.shp").to_crs(epsg=32645)
districts["area_km2"] = districts.area / 1e6

# Spatial join: which district is each point in?
joined = gpd.sjoin(points_utm, districts, predicate="within")

# Dissolve: merge districts into divisions
divisions = districts.dissolve(by="division", aggfunc="sum")

# Quick map
districts.plot(column="area_km2", cmap="viridis", legend=True, edgecolor="black")

Important GeoPandas operations:
| Operation | Meaning |
|---|---|
| `to_crs()` | Reproject |
| `buffer()` | Make zones around features |
| `sjoin()` | Spatial join (join by location) |
| `overlay()` | Intersection, union, difference of layers |
| `dissolve()` | Merge features by a column |
| `clip()` | Cut a layer with a boundary |

## **4. Rasterio: Reading and Writing Satellite Images**
Rasterio reads a GeoTIFF into a **NumPy array**, together with its georeferencing (CRS and transform), just like we did by hand on Day 92.

In [ ]:
import rasterio
import numpy as np

# Sentinel-2 bands downloaded from the Copernicus Browser
with rasterio.open("T45QXF_20240120_B04_10m.jp2") as src:
    red = src.read(1).astype("float32") / 10000
    profile = src.profile                     # CRS, transform, size ...
    print(src.crs, src.transform, src.width, src.height)

with rasterio.open("T45QXF_20240120_B08_10m.jp2") as src:
    nir = src.read(1).astype("float32") / 10000

ndvi = np.where((nir + red) > 0, (nir - red) / (nir + red), np.nan)

# Save the NDVI as a new GeoTIFF with the same georeferencing
profile.update(driver="GTiff", dtype="float32", count=1, nodata=np.nan)
with rasterio.open("kolkata_ndvi.tif", "w", **profile) as dst:
    dst.write(ndvi.astype("float32"), 1)

# Clip the image with a district boundary
from rasterio.mask import mask
# clipped, transform = mask(src, districts.geometry, crop=True)

## **5. scikit-learn: Machine Learning Classification**
Day 93's K-Means, and much more powerful **supervised** methods like **Random Forest**.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, cohen_kappa_score, confusion_matrix
from sklearn.model_selection import train_test_split

# X: one row per pixel, one column per band/index. y: the class of training pixels
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)

model = RandomForestClassifier(n_estimators=200, random_state=0)
model.fit(X_train, y_train)
predicted = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, predicted))
print("Kappa   :", cohen_kappa_score(y_test, predicted))
print(confusion_matrix(y_test, predicted))

## **6. Folium: Interactive Web Maps**
Folium makes Leaflet web maps (with zoom and pop-ups) from Python.

In [ ]:
import folium

m = folium.Map(location=[23.5, 88.0], zoom_start=7, tiles="OpenStreetMap")
for name, lat, lon in [("Kolkata", 22.5726, 88.3639), ("Siliguri", 26.7271, 88.3953), ("Digha", 21.6266, 87.5074)]:
    folium.Marker([lat, lon], popup=name).add_to(m)
folium.GeoJson("wb_weekly_weather.geojson", name="Weather").add_to(m)
m.save("west_bengal_map.html")          # Open this file in a browser

## **7. Google Earth Engine**
**Google Earth Engine (GEE)** stores petabytes of satellite data (Landsat, Sentinel, MODIS) and processes it on Google's computers, so we do not need to download anything. It is free for students and research. The `geemap` library makes it easy to use from Python.

In [ ]:
import ee, geemap
ee.Authenticate()
ee.Initialize(project="your-project-id")

kolkata = ee.Geometry.Point([88.3639, 22.5726]).buffer(20000)
image = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
         .filterBounds(kolkata)
         .filterDate("2024-01-01", "2024-02-28")
         .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 10))
         .median())
ndvi = image.normalizedDifference(["B8", "B4"]).rename("NDVI")

m = geemap.Map(center=[22.57, 88.36], zoom=10)
m.addLayer(ndvi.clip(kolkata), {"min": -0.2, "max": 0.8, "palette": ["blue", "white", "green"]}, "NDVI")
m

## **Where to Get Free Data for West Bengal**
| Data | Source |
|---|---|
| Sentinel-2, Sentinel-1 | Copernicus Browser (dataspace.copernicus.eu) |
| Landsat 8/9 | USGS EarthExplorer |
| Indian satellite data (Resourcesat, Cartosat DEM) | ISRO **Bhuvan** portal (bhuvan.nrsc.gov.in) |
| Roads, buildings, rivers | OpenStreetMap |
| District and state boundaries | Survey of India, data.gov.in, GADM |
| Weather and climate | Open-Meteo, IMD, ERA5 |

In [1]:
# Quick question
# Which library would you use to (a) calculate the area of a polygon, (b) read a GeoTIFF, (c) reproject a layer?